# KV-Cache for Autoregressive Inference

> Part of the [ML Notebooks](../README.md) series — by **Nandobez**.


## Intuition

During autoregressive decoding the keys and values for all previous tokens never change. Recomputing them every step is wasted work; instead we cache `(K, V)` for each layer and only do attention against the cached tensors plus the new token. Inference becomes $O(N)$ per token instead of $O(N^2)$.


## Mathematical Formulation

Without cache:

$$Q_t = X_t W^Q,\quad K_{1..t} = X_{1..t} W^K,\quad V_{1..t} = X_{1..t} W^V$$

With cache we just append:

$$K \leftarrow [K\,;\,X_t W^K],\qquad V \leftarrow [V\,;\,X_t W^V]$$

and recompute only the *last* query.


## Implementation


In [ ]:
import math, time, torch
import torch.nn as nn
import torch.nn.functional as F


In [ ]:
class CachedAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out = nn.Linear(d_model, d_model)

    def forward(self, x, cache=None):
        B, N, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        def split(t):
            return t.view(B, N, self.n_heads, self.d_head).transpose(1, 2)
        q, k, v = split(q), split(k), split(v)
        if cache is not None:
            k = torch.cat([cache['k'], k], dim=2)
            v = torch.cat([cache['v'], v], dim=2)
        new_cache = {'k': k, 'v': v}
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        attn = F.softmax(scores, dim=-1)
        ctx = (attn @ v).transpose(1, 2).contiguous().view(B, N, D)
        return self.out(ctx), new_cache


## Experiment


In [ ]:
# Benchmark: decode 64 tokens with and without cache
torch.manual_seed(0)
m = CachedAttention(d_model=256, n_heads=8)
m.eval()
prompt = torch.randn(1, 1, 256)

# Without cache: re-feed entire prefix each step
def naive(steps):
    x = prompt.clone()
    for _ in range(steps):
        out, _ = m(x)
        x = torch.cat([x, out[:, -1:]], dim=1)
    return x

# With cache: feed only the new token
def cached(steps):
    cache = None
    x = prompt.clone()
    out, cache = m(x, cache=None)
    for _ in range(steps):
        out, cache = m(out[:, -1:], cache=cache)
    return out

import time
with torch.no_grad():
    t0 = time.perf_counter(); naive(64); t1 = time.perf_counter()
    t2 = time.perf_counter(); cached(64); t3 = time.perf_counter()
print(f'naive  : {t1-t0:.3f}s')
print(f'cached : {t3-t2:.3f}s')
print(f'speed-up: {(t1-t0)/(t3-t2):.2f}x')


## Discussion

- The cache grows linearly in sequence length — memory is the next bottleneck.
- For batched decoding the cache must be shaped `(B, H, N, D)`; pay attention to ragged batches and paged caches (vLLM, PagedAttention).
- Quantised KV caches (int8, fp8) are the standard trick to fit longer contexts in memory.


## References

- Series repo: [github.com/Nandobez/ml-notebooks](https://github.com/Nandobez/ml-notebooks)
- Author: [Nandobez](https://github.com/Nandobez)
